#Prerequisites

##Load 'employee.csv' into DataFrame

In [0]:
employee_df=spark.read.csv(path="/Volumes/quickstart_catalog/quickstart_schema/sandbox/dataset/employee.csv",header=True,inferSchema=True, sep="|", quote="'").limit(8)
employee_df.display()

#Apply Python Function

##Create a python function

In [0]:
def convert_to_title(name):
    return name.title()

In [0]:
convert_to_title("abhay")

In [0]:
employee_df.select(convert_to_title("name")).display()
#this did not applied python function inside pyspark

#Use Python Function in Pyspark

##Convert Python function into Pyspark UDF


In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
convert_to_title_fn= udf(convert_to_title, StringType())

In [0]:
employee_df.select(convert_to_title_fn("name").alias("name")).display()

#Recommended way to create UDF

In [0]:
from pyspark.sql.functions import udf
@udf(returnType=StringType())
def convert_to_title_udf(name):
    return name.title()

In [0]:
employee_df.select(convert_to_title_udf("name").alias("name")).display()

#Handling UDF with null values

In [0]:
employee_df=spark.read.csv(path="/Volumes/quickstart_catalog/quickstart_schema/sandbox/dataset/employee.csv",header=True,inferSchema=True, sep="|", quote="'")
employee_df.display()

##Approach 1

In [0]:
from pyspark.sql.functions import udf
@udf(returnType=StringType())
def convert_to_title_udf(name):
    if name is not None:
        return name.title()
    else:
        return None

employee_df.select(convert_to_title_udf("name").alias("name")).display()

##Approach 2 (Not Working)

In [0]:
from pyspark.sql.functions import udf, when, col

@udf(returnType=StringType())
def convert_to_title_udf(name):
    return name.title()

employee_df.withColumn(
    "name",
    when(col("name").isNotNull(), convert_to_title_udf("name"))).display()

##Approach 3 (Recommended)

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
convert_to_title_fn=udf(lambda e:convert_to_title(e) if e is not None else None,StringType())

employee_df.select(convert_to_title_fn("name").alias("name")).display()